# Tahap 5 (Opsional) - Revisi & Retain
## CBR Putusan Wanprestasi PN Surabaya

**Tujuan :** Menambahkan kasus baru yang terbukti solusi-nya tepat ke dalam *case base* untuk iterasi berikutnya.

**Input:**
- `data/processed/cases.csv` → case base terkini (output Tahap 2)
- `data/processed/cases.json` → versi JSON case base
- `data/results/predictions.csv` → hasil demo manual `predict_outcome()` Tahap 4 (berisi kolom `cocok`, `top_5_case_ids`)

**Output:**
- `data/processed/cases.csv` & `cases.json` → **diperbarui** (kasus baru ditambahkan)
- `logs/retain_log.json` → riwayat retain untuk audit

**Mekanisme "terbukti tepat":**
1. **Otomatis** - filter kasus demo dengan `cocok=True` (`predicted_label == label_sebenarnya`)
2. **Override manual** - edit `QUERY_ID_DITERIMA` di Cell 3 untuk menyesuaikan secara manual

**Catatan penting:** Setelah retain, Tahap 3 dan Tahap 4 perlu **dijalankan ulang** agar kasus baru ikut masuk ke TF-IDF, SVM, dan `case_solutions` - mencerminkan siklus CBR yang sesungguhnya (iteratif).

**Urutan eksekusi:**
1. Tahap 2 → `cases.csv`
2. Tahap 3 → TF-IDF + SVM
3. Tahap 4 → `predictions.csv`
4. **Tahap 5 (ini)** → retain kasus tepat → `cases.csv` diperbarui
5. Kembali ke Tahap 3 untuk iterasi berikutnya

| Cell | Fungsi |
|---|---|
| **Cell 1** | Import & load `cases.csv` + `predictions.csv` |
| **Cell 2** | Filter otomatis kandidat retain (`cocok=True`) |
| **Cell 3** | Override manual - edit `QUERY_ID_DITERIMA` jika perlu |
| **Cell 4** | Cari kasus sumber dari `top_5_case_ids` → warisi metadata |
| **Cell 5** | Bangun baris kasus baru (format identik `cases.csv`) |
| **Cell 6** | Gabungkan ke case base → simpan `cases.csv` & `cases.json` |
| **Cell 7** | Verifikasi akhir konsistensi case base |

## Setup - Jangkar Working Directory ke Root Repository

**Jalankan cell ini SELALU sebagai cell pertama**, sebelum cell lain di notebook ini.

Notebook ini disimpan di dalam folder `notebooks/`, tapi seluruh path data di notebook ini
(mis. `'data/processed/cases.csv'`) ditulis **relatif terhadap root repository**, bukan terhadap
folder `notebooks/`. Secara default, Jupyter menjalankan kernel dengan *working directory* = folder
tempat file `.ipynb` itu sendiri berada. Tanpa cell ini, path seperti `'data/raw'` akan dicari di
`notebooks/data/raw` (tidak ada) dan menyebabkan `FileNotFoundError`.

Cell ini **aman dijalankan berkali-kali** (idempotent) — begitu working directory sudah berada di
root repository (folder yang punya subfolder `data/`), cell ini tidak akan berpindah lagi.

In [ ]:
import os

# Jika folder 'data/' tidak ada di working directory saat ini, TAPI ada satu
# level di atasnya -> kita sedang di dalam notebooks/, pindah ke root repo.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')

print('Working directory aktif :', os.getcwd())
print('Isi folder saat ini     :', sorted(os.listdir('.')))

assert os.path.isdir('data'), (
    "Folder 'data/' tidak ditemukan dari working directory ini.\n"
    "Pastikan struktur repo: <root>/notebooks/xx.ipynb dengan <root>/data/ di sebelahnya,\n"
    "dan notebook dibuka dari dalam struktur repo tersebut (bukan dipindah sendirian)."
)


## Cell 1 - Import & Load Data

In [1]:
import os
import re
import json
import pandas as pd

PATH_CASES_CSV   = 'data/processed/cases.csv'
PATH_CASES_JSON  = 'data/processed/cases.json'
PATH_PREDICTIONS = 'data/results/predictions.csv'
PATH_RETAIN_LOG  = 'logs/retain_log.json'

os.makedirs('logs', exist_ok=True)

# Load case base existing (output Tahap 2)
df_cases = pd.read_csv(PATH_CASES_CSV, encoding='utf-8-sig')

# Load hasil demo manual Tahap 4
df_pred = pd.read_csv(PATH_PREDICTIONS, encoding='utf-8-sig')

print('=' * 62)
print('  TAHAP 5 (OPSIONAL) — REVISI & RETAIN')
print('=' * 62)
print(f'  Case base saat ini     : {len(df_cases)} kasus')
print(f'  Kolom case base        : {list(df_cases.columns)}')
print(f'  Hasil demo Tahap 4     : {len(df_pred)} kasus diuji')
print(f'  Kolom predictions.csv  : {list(df_pred.columns)}')
print(f'  Cocok (prediksi=label) : {df_pred["cocok"].sum()} / {len(df_pred)}')
print('=' * 62)

  TAHAP 5 (OPSIONAL) — REVISI & RETAIN
  Case base saat ini     : 54 kasus
  Kolom case base        : ['case_id', 'no_perkara', 'tanggal', 'jenis_perkara', 'pengadilan', 'pihak', 'pasal', 'ringkasan_fakta', 'argumen_hukum', 'amar_putusan', 'amar_lengkap', 'word_count', 'top_terms', 'label_putusan', 'text_full', 'cek_pihak']
  Hasil demo Tahap 4     : 5 kasus diuji
  Kolom predictions.csv  : ['query_id', 'query', 'predicted_solution', 'top_5_case_ids', 'predicted_label', 'predicted_label_nama', 'label_sebenarnya', 'cocok']
  Cocok (prediksi=label) : 4 / 5


## Cell 2 - Filter Otomatis Kandidat Retain

"Solusi yang terbukti tepat" = `predicted_label` hasil `predict_outcome()` di Tahap 4 **sama** dengan `label_sebenarnya` yang ditetapkan saat menyusun 5 kasus demo manual.

In [2]:
# ── Filter otomatis: kasus dengan prediksi TEPAT (cocok == True) ────────────
# "Solusi yang terbukti tepat" diartikan sebagai: predicted_label sama
# dengan label_sebenarnya (hasil bandingan manual di Tahap 4 Cell 7).

kandidat_retain = df_pred[df_pred['cocok'] == True].copy()

print(f'Kandidat retain otomatis (cocok=True): {len(kandidat_retain)} dari {len(df_pred)} kasus demo')
print()

if len(kandidat_retain) == 0:
    print('Tidak ada kasus yang cocok pada demo manual saat ini.')
    print('→ Retain dilewati untuk iterasi ini, case base tetap seperti semula.')
else:
    for _, row in kandidat_retain.iterrows():
        print(f"  [{row['query_id']}] {row['predicted_label_nama']}")
        print(f"     query         : {row['query'][:90]}...")
        print(f"     top_5_case_ids: {row['top_5_case_ids']}")
    print()
    print('Daftar query_id yang akan di-retain (default = semua kandidat di atas):')
    print(f"  {kandidat_retain['query_id'].tolist()}")


Kandidat retain otomatis (cocok=True): 4 dari 5 kasus demo

  [D001] Ditolak/NO
     query         : tergugat sebagai kontraktor tidak menyelesaikan pekerjaan renovasi rumah sesuai RAB dan pe...
     top_5_case_ids: case_002, case_036, case_004, case_038, case_001
  [D002] Dikabulkan
     query         : penggugat memberikan pinjaman uang kepada tergugat namun tergugat tidak mengembalikan sesu...
     top_5_case_ids: case_039, case_022, case_026, case_051, case_008
  [D003] Ditolak/NO
     query         : gugatan diajukan ke pengadilan negeri surabaya namun objek sengketa berada di luar wilayah...
     top_5_case_ids: case_014, case_042, case_045, case_047, case_054
  [D005] Ditolak/NO
     query         : penggugat mengajukan gugatan wanprestasi terhadap tergugat namun penggugat sendiri belum m...
     top_5_case_ids: case_021, case_042, case_039, case_002, case_025

Daftar query_id yang akan di-retain (default = semua kandidat di atas):
  ['D001', 'D002', 'D003', 'D005']


## Cell 3 - Override Manual (Opsional)

Edit `QUERY_ID_DITERIMA` di cell ini jika ingin:
- **Mengecualikan** kasus tertentu meski `cocok=True` (hapus dari list)
- **Menambahkan** kasus `cocok=False` yang menurut Anda tetap layak dijadikan referensi

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# OVERRIDE MANUAL — sesuaikan list ini sebelum lanjut ke cell berikutnya.
#
# Defaultnya: semua kandidat otomatis (cocok=True) akan di-retain.
# Edit QUERY_ID_DITERIMA di bawah untuk:
#   - Mengecualikan kasus tertentu meski cocok=True (hapus dari list)
#   - Menambahkan kasus tertentu meski cocok=False, jika menurut Anda
#     solusinya tetap layak dijadikan referensi (tambahkan manual ke list)
# ═══════════════════════════════════════════════════════════════════════

QUERY_ID_DITERIMA = kandidat_retain['query_id'].tolist()  # <-- EDIT DI SINI JIKA PERLU

# Contoh override manual (uncomment & sesuaikan jika perlu):
# QUERY_ID_DITERIMA = ['D002', 'D003']          # hanya retain 2 kasus ini
# QUERY_ID_DITERIMA = kandidat_retain['query_id'].tolist() + ['D004']  # tambah D004 manual

df_final_retain = df_pred[df_pred['query_id'].isin(QUERY_ID_DITERIMA)].copy()

print(f'Kasus yang akan benar-benar di-retain: {len(df_final_retain)}')
for _, row in df_final_retain.iterrows():
    sumber = 'otomatis (cocok)' if row['cocok'] else '✋ override manual'
    print(f"  [{row['query_id']}] {row['predicted_label_nama']} — {sumber}")


Kasus yang akan benar-benar di-retain: 4
  [D001] Ditolak/NO — otomatis (cocok)
  [D002] Dikabulkan — otomatis (cocok)
  [D003] Ditolak/NO — otomatis (cocok)
  [D005] Ditolak/NO — otomatis (cocok)


## Cell 4 - Cari Kasus Sumber dari `top_5_case_ids`

**Bagian yang diperbaiki.** Setiap query demo (`D001`–`D005`) punya kolom `top_5_case_ids` (hasil `retrieve()` Tahap 3/4) - daftar `case_id` di case base yang paling mirip dengan query tersebut. Karena solusi (`predicted_solution`) memang **diwariskan** dari kasus-kasus ini (lihat `predict_outcome()` di Tahap 4 - `majority_vote()`/`weighted_similarity()` atas `top_k`), maka metadata hukum seperti `pasal`, `pihak`, dan `no_perkara` juga seharusnya diwariskan dari kasus sumber yang sama, **bukan** diekstrak ulang dari kalimat query.

Yang diambil dari kasus sumber (`case_id` pertama pada `top_5_case_ids`, yaitu kasus paling mirip):
- `pasal` - referensi pasal yang sudah diekstrak dengan benar di Tahap 2 dari teks putusan asli
- `pihak` - Penggugat vs. Tergugat
- `no_perkara` - sebagai referensi (`no_perkara_acuan`), bukan menggantikan penanda `RETAIN-xxx`
- `argumen_hukum` - cuplikan singkat sebagai konteks tambahan

Jika karena suatu hal `top_5_case_ids` kosong/tidak valid, fallback otomatis ke `ekstrak_pasal_dari_teks()` pada kalimat query (perilaku versi lama) supaya pipeline tidak gagal — namun ini idealnya tidak pernah terjadi karena `retrieve()` selalu mengembalikan k>0 hasil untuk corpus yang tidak kosong.

In [4]:
def ekstrak_pasal_fallback(teks: str) -> str:
    """
    Fallback: ekstraksi pasal langsung dari teks query.
    Dipakai HANYA jika kasus sumber tidak ditemukan di case base.
    """
    matches = re.findall(
        r'pasal\s+(\d+(?:\s*(?:ayat|huruf|dan|jo\.?|juncto)?\s*\d*)*)',
        teks, re.IGNORECASE
    )
    pasal_list, seen = [], set()
    for m in matches:
        nomor = re.sub(r'\s+', ' ', m.strip())
        if re.search(r'\d{3,}', nomor):
            label = f'Pasal {nomor}'
            if label not in seen:
                pasal_list.append(label)
                seen.add(label)
    return ', '.join(pasal_list) if pasal_list else 'Tidak Tertera'


def cari_kasus_sumber(top_5_str: str, df_case_base: pd.DataFrame) -> dict:
    """
    Ambil metadata dari case_id PERTAMA (paling mirip) di top_5_case_ids.
    top_5_str format: 'case_001, case_017, ...' (dipisah koma+spasi)

    Returns:
        dict — {case_id_sumber, pasal, pihak, no_perkara, argumen_hukum}
        None — jika tidak ada case_id valid ditemukan di case base
    """
    if not isinstance(top_5_str, str) or not top_5_str.strip():
        return None

    daftar_id = [cid.strip() for cid in top_5_str.split(',') if cid.strip()]

    for cid in daftar_id:
        baris = df_case_base[df_case_base['case_id'] == cid]
        if len(baris) > 0:
            r = baris.iloc[0]
            return {
                'case_id_sumber': cid,
                'pasal'         : r.get('pasal', 'Tidak Tertera'),
                'pihak'         : r.get('pihak', 'Tidak Tertera'),
                'no_perkara'    : r.get('no_perkara', 'Tidak Tertera'),
                'argumen_hukum' : r.get('argumen_hukum', ''),
            }
    return None


# Cari kasus sumber untuk setiap kandidat retain
print('=' * 62)
print('  PENCARIAN KASUS SUMBER (top_5_case_ids → metadata)')
print('=' * 62)
print()

info_sumber = {}
for _, row in df_final_retain.iterrows():
    qid    = row['query_id']
    sumber = cari_kasus_sumber(row['top_5_case_ids'], df_cases)

    if sumber is not None:
        info_sumber[qid] = sumber
        print(f"  [{qid}] kasus sumber : {sumber['case_id_sumber']}")
        print(f"         pasal warisan : {sumber['pasal']}")
        print(f"         pihak sumber  : {sumber['pihak']}")
    else:
        pasal_fb = ekstrak_pasal_fallback(row['query'])
        info_sumber[qid] = {
            'case_id_sumber': 'TIDAK_DITEMUKAN',
            'pasal'         : pasal_fb,
            'pihak'         : 'Tidak Tertera',
            'no_perkara'    : 'Tidak Tertera',
            'argumen_hukum' : '',
        }
        print(f"  [{qid}] ⚠️  kasus sumber tidak ditemukan → fallback ekstraksi dari query")
        print(f"         pasal fallback : {pasal_fb}")
    print()

n_punya_pasal = sum(1 for v in info_sumber.values() if v['pasal'] != 'Tidak Tertera')
print('=' * 62)
print(f'  Ringkasan: {n_punya_pasal}/{len(info_sumber)} kasus retain berhasil mewarisi pasal')
print('=' * 62)

  PENCARIAN KASUS SUMBER (top_5_case_ids → metadata)

  [D001] kasus sumber : case_002
         pasal warisan : Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332
         pihak sumber  : Kandar Panjaitan vs. Johan Sugiharto Se

  [D002] kasus sumber : case_039
         pasal warisan : Pasal 118 ayat 1, Pasal 123, Pasal 1234 jo, Pasal 1238, Pasal 1234, Pasal 1246, Pasal 1248, Pasal 1250, Pasal 1248 dan, Pasal 1248dan, Pasal 180, Pasal 1243, Pasal 1267, Pasal 606, Pasal 180 1
         pihak sumber  : Santi Boediyanto vs. 1 Go Gorry Lisayati

  [D003] kasus sumber : case_014
         pasal warisan : Pasal 1266, Pasal 1267, Pasal 180 ayat 1, Pasal 1917, Pasal 378, Pasal 372, Pasal 1517, Pasal 1266 dan1267, Pasal 1266 ayat 1, Pasal 1266 ayat 2, Pasal 118, Pasal 136
         pihak sumber  : 1 Oei Ie Ling vs. 1 Donny Sanjaya

  [D005] kasus sumber : case_021
         pasal warisan : Pasal 1338, Pasal 1243, Pasal 181 ayat 2,

# DEBUG

In [5]:
# Debug: lihat 400 karakter awal dari kedua case yang bermasalah
for cid in ['case_039', 'case_014']:
    with open(f'data/raw/{cid}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()
    print(f"=== {cid} ===")
    print(teks[:400])
    print()

=== case_039 ===
putusan nomor 143 pdt g 2025 pn sby demi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang mengadili perkara perkara perdatapada peradilan tingkat pertama telah menjatuhkan putusan sebagai berikut dalamperkara antara santi boediyanto tempat tanggal lahir surabaya 13 09 1968 umur 56 tahun jenis kelamin perempuan agama islam warga negara indonesia alamat apartemen rose baya

=== case_014 ===
p u t u s a nnomor 1358 pdt g 2023 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang memeriksa dan memutus perkaraperdata pada tingkat pertama telah menjatuhkan putusan sebagai berikut dalamperkara gugatan antara 1 oei ie ling tempat tanggal lahir malang 27 november 1951 umur72 tahun jenis kelamin perempuan agama budga warga negara indonesia pekerjaan wiraswast



In [6]:
# Debug: lihat bagian sekitar kata "lawan"/"melawan" pada kedua case yang bermasalah
for cid in ['case_039', 'case_014']:
    with open(f'data/raw/{cid}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()

    # Cari posisi kata "lawan" pertama dalam teks
    import re
    m = re.search(r'\b(?:me)?lawan\b', teks, re.IGNORECASE)
    print(f"=== {cid} ===")
    if m:
        mulai = max(0, m.start() - 20)
        selesai = min(len(teks), m.start() + 250)
        print(f"Posisi 'lawan' ditemukan di karakter ke-{m.start()}")
        print(teks[mulai:selesai])
    else:
        print("Kata 'lawan'/'melawan' TIDAK DITEMUKAN sama sekali dalam dokumen ini!")
    print()

=== case_039 ===
Posisi 'lawan' ditemukan di karakter ke-1029
t sebagai penggugat lawan 1 go gorry lisayati tempat tanggal lahir surabaya o8 02 1978 umur 46 tahun jenis kelamin perempuan agama islam warga negara indonesia alamat jl tanjungtorawitan no 23 25 rt 006 rw 008 kel perak barat kec krembangan surabaya pekerjaaan mengurusr

=== case_014 ===
Kata 'lawan'/'melawan' TIDAK DITEMUKAN sama sekali dalam dokumen ini!



In [7]:
with open('data/raw/case_014.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

print(f"Total panjang teks: {len(teks)} karakter")
print()
print("--- 1000 karakter pertama ---")
print(teks[:1000])

Total panjang teks: 124597 karakter

--- 1000 karakter pertama ---
p u t u s a nnomor 1358 pdt g 2023 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang memeriksa dan memutus perkaraperdata pada tingkat pertama telah menjatuhkan putusan sebagai berikut dalamperkara gugatan antara 1 oei ie ling tempat tanggal lahir malang 27 november 1951 umur72 tahun jenis kelamin perempuan agama budga warga negara indonesia pekerjaan wiraswasta alamat jl manyar kertoarjo 7 12 rt 005 rw 011 kelurahan mojo kecamatan gubeng surabaya selanjutnya disebut sebagai penggugat i 2 william darwoko tempat tanggal lahir surabaya 17 september1987 umur 36 tahun jenis kelamin laki laki agamakristen warga negara indonesia pekerjaan wiraswasta alamat jl embong blimbing no 24 rt 002 rw 008 kelurahan embong kaliasin kecamatan genteng surabaya selanjutnya disebut sebagai penggugat ii dalam hal ini penggugat i dan penggugat ii memberikan kuasa kepadamohammad asikin s h advokat pengacara 

## Cell 5 — Bangun Baris Kasus Baru

`case_id` baru otomatis melanjutkan nomor terakhir di case base (contoh: jika case base berakhir di `case_054`, kasus retain pertama menjadi `case_055`). Format kolom identik dengan `cases.csv` agar kompatibel dengan pipeline Tahap 3 dan 4.

In [8]:
def case_id_berikutnya(df_existing: pd.DataFrame, n_baru: int) -> list:
    """
    Generate case_id baru yang melanjutkan urutan numerik terakhir di case base.
    Contoh: case base berakhir case_054 → return ['case_055', 'case_056', ...]
    """
    existing_ids   = df_existing['case_id'].tolist()
    nomor_existing = []
    for cid in existing_ids:
        m = re.search(r'\d+', str(cid))
        if m:
            nomor_existing.append(int(m.group()))
    nomor_terakhir = max(nomor_existing) if nomor_existing else 0
    return [f'case_{nomor_terakhir + i:03d}' for i in range(1, n_baru + 1)]


new_ids = case_id_berikutnya(df_cases, len(df_final_retain))

baris_baru = []
for new_id, (_, row) in zip(new_ids, df_final_retain.iterrows()):
    qid        = row['query_id']
    query_teks = row['query']
    solusi     = row['predicted_solution']
    label      = row['predicted_label']   # label yang TERBUKTI tepat
    sumber     = info_sumber[qid]

    baris_baru.append({
        'case_id'        : new_id,
        'no_perkara'     : f'RETAIN-{qid}',   # penanda: kasus hasil retain
        'tanggal'        : pd.Timestamp.now().strftime('%Y-%m-%d'),
        'ringkasan_fakta': query_teks,
        'pasal'          : sumber['pasal'],    # diwariskan dari kasus sumber
        'pihak'          : sumber['pihak'],    # diwariskan dari kasus sumber
        'argumen_hukum'  : (
            f"(retained) Solusi & pasal diadopsi dari kasus sumber "
            f"{sumber['case_id_sumber']} (top-1 retrieve()). "
            f"Acuan no_perkara sumber: {sumber['no_perkara']}. "
            + (f"Cuplikan argumen kasus sumber: {str(sumber['argumen_hukum'])[:200]}..."
               if sumber['argumen_hukum'] else "")
        ),
        'amar_lengkap'   : solusi,
        'word_count'     : len(query_teks.split()) + len(str(solusi).split()),
        'top_terms'      : '',  # dihitung ulang saat re-fit TF-IDF di Tahap 3
        'label_putusan'  : label,
        'text_full'      : query_teks + ' ' + str(solusi),
    })

df_retain_baru = pd.DataFrame(baris_baru)

print(f'{len(df_retain_baru)} kasus baru siap ditambahkan ke case base:')
print()
for _, row in df_retain_baru.iterrows():
    print(f"  {row['case_id']}  [{row['no_perkara']}]  label={row['label_putusan']}")
    print(f"    pasal           : {str(row['pasal'])[:80]}")
    print(f"    pihak           : {row['pihak']}")
    print(f"    ringkasan_fakta : {row['ringkasan_fakta'][:80]}...")
    print()

4 kasus baru siap ditambahkan ke case base:

  case_055  [RETAIN-D001]  label=0
    pasal           : Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 
    pihak           : Kandar Panjaitan vs. Johan Sugiharto Se
    ringkasan_fakta : tergugat sebagai kontraktor tidak menyelesaikan pekerjaan renovasi rumah sesuai ...

  case_056  [RETAIN-D002]  label=1
    pasal           : Pasal 118 ayat 1, Pasal 123, Pasal 1234 jo, Pasal 1238, Pasal 1234, Pasal 1246, 
    pihak           : Santi Boediyanto vs. 1 Go Gorry Lisayati
    ringkasan_fakta : penggugat memberikan pinjaman uang kepada tergugat namun tergugat tidak mengemba...

  case_057  [RETAIN-D003]  label=0
    pasal           : Pasal 1266, Pasal 1267, Pasal 180 ayat 1, Pasal 1917, Pasal 378, Pasal 372, Pasa
    pihak           : 1 Oei Ie Ling vs. 1 Donny Sanjaya
    ringkasan_fakta : gugatan diajukan ke pengadilan negeri surabaya namun objek sengketa berada di lu...

  case_058  [RETAIN-D005]  label=0
    

## Cell 6 - Simpan ke Case Base (`cases.csv` & `cases.json`)

In [9]:
if len(df_retain_baru) == 0:
    print('Tidak ada kasus baru untuk di-retain pada iterasi ini.')
    print('cases.csv & cases.json tidak diubah.')
else:
    sebelum = len(df_cases)

    # Gabungkan — pastikan urutan kolom tetap identik dengan case base asli
    df_cases_updated = pd.concat([df_cases, df_retain_baru], ignore_index=True)
    df_cases_updated = df_cases_updated.reindex(columns=df_cases.columns)

    sesudah = len(df_cases_updated)

    # Simpan cases.csv
    df_cases_updated.to_csv(PATH_CASES_CSV, index=False, encoding='utf-8-sig')

    # Simpan cases.json (tanpa text_full agar file ringkas)
    records = df_cases_updated.drop(columns=['text_full'], errors='ignore').to_dict(orient='records')
    with open(PATH_CASES_JSON, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    # Simpan retain_log.json (append, bukan overwrite — jaga riwayat iterasi)
    log_retain = {
        'timestamp'               : pd.Timestamp.now().isoformat(),
        'jumlah_ditambah'         : len(df_retain_baru),
        'case_id_baru'            : df_retain_baru['case_id'].tolist(),
        'sumber_query_id'         : df_final_retain['query_id'].tolist(),
        'case_id_sumber_pasal'    : {qid: info_sumber[qid]['case_id_sumber']
                                     for qid in df_final_retain['query_id']},
        'case_base_sebelum'       : sebelum,
        'case_base_sesudah'       : sesudah,
    }
    log_history = []
    if os.path.exists(PATH_RETAIN_LOG):
        with open(PATH_RETAIN_LOG, 'r', encoding='utf-8') as f:
            log_history = json.load(f)
    log_history.append(log_retain)
    with open(PATH_RETAIN_LOG, 'w', encoding='utf-8') as f:
        json.dump(log_history, f, ensure_ascii=False, indent=2)

    print('=' * 62)
    print('  RETAIN SELESAI')
    print('=' * 62)
    print(f'  Case base sebelum  : {sebelum} kasus')
    print(f'  Case base sesudah  : {sesudah} kasus  (+{sesudah - sebelum})')
    print(f'  case_id baru       : {df_retain_baru["case_id"].tolist()}')
    print()
    print(f'  {PATH_CASES_CSV} diperbarui')
    print(f'  {PATH_CASES_JSON} diperbarui')
    print(f'  {PATH_RETAIN_LOG} dicatat')
    print('=' * 62)
    print()
    print('     PENTING untuk iterasi berikutnya:')
    print('  Tahap 3 (TF-IDF + SVM) dan Tahap 4 (case_solutions) perlu')
    print('  dijalankan ULANG agar kasus baru ikut masuk ke model.')
    print('  cases.csv yang diperbarui ini menjadi case base baru')
    print('  untuk siklus CBR selanjutnya.')

  RETAIN SELESAI
  Case base sebelum  : 54 kasus
  Case base sesudah  : 58 kasus  (+4)
  case_id baru       : ['case_055', 'case_056', 'case_057', 'case_058']

  data/processed/cases.csv diperbarui
  data/processed/cases.json diperbarui
  logs/retain_log.json dicatat

     PENTING untuk iterasi berikutnya:
  Tahap 3 (TF-IDF + SVM) dan Tahap 4 (case_solutions) perlu
  dijalankan ULANG agar kasus baru ikut masuk ke model.
  cases.csv yang diperbarui ini menjadi case base baru
  untuk siklus CBR selanjutnya.


## Cell 7 - Verifikasi Akhir

In [10]:
df_cek = pd.read_csv(PATH_CASES_CSV, encoding='utf-8-sig')

print('=' * 62)
print('  VERIFIKASI CASE BASE SETELAH RETAIN')
print('=' * 62)
print(f'  Total kasus              : {len(df_cek)}')
print(f'  case_id unik?            : {df_cek["case_id"].is_unique}')
print(f'  Kolom tetap konsisten?   : '
      f'{list(df_cek.columns) == list(df_cases.columns) if len(df_retain_baru) > 0 else "N/A"}')
print()
print('  Distribusi label putusan:')
dist = df_cek['label_putusan'].value_counts().sort_index()
label_nama = {0: 'Ditolak/NO', 1: 'Dikabulkan', 2: 'Damai', -1: 'Unknown'}
for lbl, cnt in dist.items():
    print(f'    {label_nama.get(lbl, str(lbl)):<25} (label {lbl}): {cnt} kasus')
print('=' * 62)

if len(df_retain_baru) > 0:
    print()
    print('  Preview kasus hasil retain:')
    cols_preview = ['case_id', 'no_perkara', 'pasal', 'pihak', 'label_putusan']
    cols_ada = [c for c in cols_preview if c in df_cek.columns]
    df_retained_preview = df_cek[df_cek['case_id'].isin(df_retain_baru['case_id'])]
    print(df_retained_preview[cols_ada].to_string(index=False))
    print()

    n_kosong = (df_retain_baru['pasal'] == 'Tidak Tertera').sum()
    n_terisi = len(df_retain_baru) - n_kosong
    print('  Cek kolom pasal pada kasus hasil retain:')
    print(f'    Terisi (diwariskan dari kasus sumber) : {n_terisi} / {len(df_retain_baru)}')
    print(f'    Masih "Tidak Tertera" (fallback)      : {n_kosong} / {len(df_retain_baru)}')

    print()
    print('  Log retain tersimpan di:', PATH_RETAIN_LOG)
    with open(PATH_RETAIN_LOG, 'r', encoding='utf-8') as f:
        log = json.load(f)
    last = log[-1]
    print(f'    Iterasi terakhir  : {last["timestamp"]}')
    print(f'    Ditambahkan       : {last["jumlah_ditambah"]} kasus')
    print(f'    case_id baru      : {last["case_id_baru"]}')

print()
print('=' * 62)
print('  STATUS: TAHAP 5 SELESAI!')
print('  Lanjut ke Tahap 6 (Model Evaluation).')
print('=' * 62)

  VERIFIKASI CASE BASE SETELAH RETAIN
  Total kasus              : 58
  case_id unik?            : True
  Kolom tetap konsisten?   : True

  Distribusi label putusan:
    Ditolak/NO                (label 0): 27 kasus
    Dikabulkan                (label 1): 29 kasus
    Damai                     (label 2): 2 kasus

  Preview kasus hasil retain:
 case_id  no_perkara                                                                                                                                                                                            pasal                                    pihak  label_putusan
case_055 RETAIN-D001                                                                          Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332  Kandar Panjaitan vs. Johan Sugiharto Se              0
case_056 RETAIN-D002 Pasal 118 ayat 1, Pasal 123, Pasal 1234 jo, Pasal 1238, Pasal 1234, Pasal 1246, Pasal 1248, 